# Baseline DSA via `lic_dsf.dsa`

Build Ext + Macro books, then external/public sustainability ratios
(Output 1-1 / 1-2 engines).

See `docs/07-baseline-dsa.md`.

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from lic_dsf.dsa import (
    BaselineExternalBook,
    BaselinePublicBook,
    external_dsa_panel,
    public_dsa_panel,
)
from lic_dsf.pv import (
    ExternalDebtBook,
    MacroDebtBook,
    PVPortfolio,
    load_external_debt_inputs,
    load_instruments_from_workbook,
    load_lc_nr_instruments_from_workbook,
    load_macro_debt_inputs,
)

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "demo":
    REPO_ROOT = REPO_ROOT.parent

WORKBOOK = REPO_ROOT / "data" / "lic-dsf-template-2025-08-12.xlsx"

pd.set_option("display.max_columns", 24)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

WORKBOOK

In [ ]:
instruments = load_instruments_from_workbook(
    WORKBOOK, include_zero_disbursement=True
)
lc_nr = load_lc_nr_instruments_from_workbook(
    WORKBOOK, include_zero_disbursement=True
)
ext = ExternalDebtBook(
    portfolio=PVPortfolio(instruments=tuple(instruments) + tuple(lc_nr)),
    inputs=load_external_debt_inputs(WORKBOOK),
)
macro = MacroDebtBook(inputs=load_macro_debt_inputs(WORKBOOK), external=ext)
ext_base = BaselineExternalBook(macro=macro, external=ext)
pub_base = BaselinePublicBook(macro=macro, external=ext)
years = [2022, 2023, 2024, 2025, 2026]
ext_base.years[0], ext_base.years[-1], macro.inputs.first_projection_year

## External DSA (Output 1-1 shape)

In [ ]:
external_dsa_panel(ext_base).loc[:, years]

## Public DSA (Output 1-2 shape)

In [ ]:
public_dsa_panel(pub_base).loc[:, years]